# NB-sweep — backbone/decoder screen (does a transformer beat U-Net?)

Screens 7 encoder/decoder combos as early-fusion multimodal nets on the SAME
split + recipe (curriculum + consistency + Focal-Tversky + EMA). **1 seed each**
to find the ceiling cheaply, incl. three lightweight **transformers** (SegFormer /
MiT). Then 3-seed only the winner.

Reference: unet+mobilenet_v2 (= unet_drop) = 0.801 both / 0.598 s1 / 0.781 s2.

**Attach:** `lgmf_c` (v7), Sen1Floods11, `lgmf-index`. GPU + Internet on. ~2.5h (7 runs).


In [ ]:
import sys, os, glob, json, random, time, warnings
warnings.filterwarnings("ignore")
init = glob.glob("/kaggle/input/**/lgmf/__init__.py", recursive=True)
assert init, "attach lgmf_c v7"
sys.path.insert(0, os.path.dirname(os.path.dirname(init[0])))
try:
    import segmentation_models_pytorch  # noqa
except ImportError:
    os.system("pip install -q segmentation-models-pytorch")
import numpy as np, torch
from torch.utils.data import DataLoader
import lgmf
from lgmf.config import LGMFConfig
from lgmf.dataset import Sen1Floods11
from lgmf.baselines import build_baseline
from lgmf.model import count_params
from lgmf.ema import ModelEMA
from lgmf.engine import train_one_epoch, evaluate
print("lgmf", lgmf.__version__)
idx = os.path.dirname(glob.glob("/kaggle/input/**/norm_stats.json", recursive=True)[0])
stats = json.load(open(f"{idx}/norm_stats.json"))

In [ ]:
SEED, EPOCHS = 42, 45
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin = device.type == "cuda"; regs = ["both", "s1_only", "s2_only"]

# (name, arch, encoder). 'T' marks transformer backbones.
SWEEP = [
    ("unet_mobilenet",   "unet",         "mobilenet_v2"),     # = unet_drop reference
    ("unet_effb0",       "unet",         "efficientnet-b0"),
    ("unet_mit",         "unet",         "mit_b0"),           # T
    ("segformer_mit",    "segformer",    "mit_b0"),           # T (3.7M, lightest)
    ("manet_mit",        "manet",        "mit_b0"),           # T
    ("fpn_mit",          "fpn",          "mit_b0"),           # T
    ("unetpp_effb0",     "unetplusplus", "efficientnet-b0"),
]

def make_cfg(name):
    c = LGMFConfig()
    c.epochs, c.lr, c.batch_size = EPOCHS, 3e-4, 8
    c.ema_decay, c.loss_type, c.consistency_weight = 0.99, "focal_tversky", 0.5
    c.num_workers = 0
    c.split_dir, c.ckpt_dir = idx, "/kaggle/working/ckpt_sweep"
    c.s1_mean, c.s1_std = stats["s1_mean"], stats["s1_std"]
    c.s2_mean, c.s2_std = stats["s2_mean"], stats["s2_std"]
    c.seed, c.run_name = SEED, name
    return c

def seed_all(sd):
    random.seed(sd); np.random.seed(sd); torch.manual_seed(sd); torch.cuda.manual_seed_all(sd)
    torch.backends.cudnn.benchmark = False

def run_one(cfg, arch, enc):
    seed_all(cfg.seed)
    tr = DataLoader(Sen1Floods11(cfg, f"{idx}/train_index.json", augment=True),
                    batch_size=cfg.batch_size, shuffle=True, num_workers=0, pin_memory=pin, drop_last=True)
    va = DataLoader(Sen1Floods11(cfg, f"{idx}/val_index.json", augment=False),
                    batch_size=cfg.batch_size, shuffle=False, num_workers=0, pin_memory=pin)
    model = build_baseline(arch=arch, encoder=enc, in_ch=8, pretrained=True).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs)
    scaler = torch.amp.GradScaler("cuda") if (cfg.amp and device.type=="cuda") else None
    ema = ModelEMA(model, cfg.ema_decay); em = ema.module
    best, bv = -1.0, None
    for epoch in range(cfg.epochs):
        train_one_epoch(model, tr, opt, cfg, epoch, device, scaler=scaler, ema=ema)
        sched.step()
        val = evaluate(em, va, cfg, device)
        sc = float(np.mean([val[r]["iou"] for r in regs]))
        if sc > best:
            best = sc
            bv = {r: {"iou": val[r]["iou"]} for r in regs}
            D = os.path.join(cfg.ckpt_dir, cfg.run_name); os.makedirs(D, exist_ok=True)
            torch.save({"model": model.state_dict(), "ema": ema.module.state_dict(),
                        "baseline": {"in_ch": 8, "arch": arch, "encoder": enc},
                        "val": bv, "cfg": dict(cfg.__dict__)}, os.path.join(D, "best.pt"))
    return best, bv, count_params(model)

## Screen all combos (1 seed)

In [ ]:
results = {}
t0 = time.time()
for name, arch, enc in SWEEP:
    D = os.path.join("/kaggle/working/ckpt_sweep", name)
    rj = os.path.join(D, "result.json")
    if os.path.exists(rj):
        rec = json.load(open(rj)); results[name] = rec; print(f"[skip] {name}")
        continue
    best, bv, params = run_one(make_cfg(name), arch, enc)
    rec = {"best": best, "val": bv, "params": params, "arch": arch, "encoder": enc}
    os.makedirs(D, exist_ok=True); json.dump(rec, open(rj,"w"), indent=2, default=str)
    results[name] = rec
    print(f"[done] {name:16s} {params:5.2f}M  both {bv['both']['iou']:.3f} "
          f"s1 {bv['s1_only']['iou']:.3f} s2 {bv['s2_only']['iou']:.3f}  ({(time.time()-t0)/60:.0f} min)")
print(f"\nsweep done in {(time.time()-t0)/60:.0f} min")

## Ranked table

In [ ]:
rows = []
for name, arch, enc in SWEEP:
    if name not in results: continue
    r = results[name]; v = r["val"]
    mean = np.mean([v[k]["iou"] for k in regs])
    rows.append((name, r["params"], v["both"]["iou"], v["s1_only"]["iou"], v["s2_only"]["iou"], mean))
rows.sort(key=lambda x: -x[5])
print("=" * 82)
print(f"{'model':16s} {'params':>7s} {'both':>8s} {'s1_only':>8s} {'s2_only':>8s} {'mean':>8s}")
print("-" * 82)
for name, p, b, s1, s2, m in rows:
    star = "  <-- unet_drop ref" if name == "unet_mobilenet" else ""
    print(f"{name:16s} {p:6.2f}M {b:8.3f} {s1:8.3f} {s2:8.3f} {m:8.3f}{star}")
print("=" * 82)
print("1-seed screen (noisy ~0.02). Take the top 1-2 to a 3-seed confirmation run.")
print("A transformer (mit/segformer) beating unet_mobilenet on 'both' AND 'mean' = worth pursuing.")